# Notebook 05: Fine-Tuning Setup

**Purpose:** Setup fine-tuning pipeline for PaddleOCR-VL (actual training in next phase).

In [ ]:
import sys
sys.path.append('./docuspend/src')

from pathlib import Path
import yaml
import json
import os
from train import TrainingSetup, TrainingConfig

base_dir = Path("/content/docuspend") if os.path.exists("/content") else Path("./docuspend")
print(f"Working with base directory: {base_dir}")

In [ ]:
# Step 1: Load pretrained model
print("Loading PaddleOCR-VL model...")
try:
    from paddleocr import PaddleOCR
    ocr = PaddleOCR(lang="en", use_gpu=True)
    print("✓ PaddleOCR-VL model loaded")
except Exception as e:
    print(f"Note: PaddleOCR load skipped (running verification only): {e}")

In [ ]:
# Step 2: Load training config
config_path = base_dir / "configs/training_config.yaml"
config = TrainingConfig(str(config_path))

print("\nTraining Configuration:")
for key in ["training", "optimization", "hardware"]:
    print(f"  {key}:")
    for sub_k, sub_v in config.get(key, {}).items():
        print(f"    {sub_k}: {sub_v}")

In [ ]:
# Step 3: Setup training pipeline
training_setup = TrainingSetup(config, str(base_dir / "outputs"))
setup_logger = training_setup.setup_logging()

print("✓ Training directories created")
print("✓ Logging configured")

In [ ]:
# Step 4: Load preprocessed data info
from dataset import ReceiptDataset

train_dataset = ReceiptDataset(
    str(base_dir / "data/processed/train"),
    str(base_dir / "data/annotations/train"),
    split="train",
    image_format="png"
)

val_dataset = ReceiptDataset(
    str(base_dir / "data/processed/val"),
    str(base_dir / "data/annotations/val"),
    split="val",
    image_format="png"
)

print(f"\nDataset loaded:")
print(f"  Train: {len(train_dataset)} images")
print(f"  Val: {len(val_dataset)} images")

In [ ]:
# Step 5 & 6: Verification and output
verification = training_setup.verify_setup()
device_info = verification["device_info"]
training_info = verification["training_info"]

print("="*60)
print("FINE-TUNING SETUP VERIFICATION")
print("="*60)
print(f"\nDevice Information:")
print(f"  Device: {device_info.get('device')}")
print(f"  GPU Available: {device_info.get('cuda_available')}")
if device_info.get('gpu_name'):
    print(f"  GPU Name: {device_info.get('gpu_name')}")
    print(f"  GPU Memory: {device_info.get('gpu_memory_mb')} MB")

print(f"\nTraining Configuration:")
for k, v in training_info.items():
    print(f"  {k}: {v}")

print(f"\n✅ Training pipeline configured and ready")
print(f"✅ To start training, run: python src/train.py")
print(f"✅ Or continue with Notebook 06 for inference testing")